In [ ]:
# after merge do the feature removal and  selection

# Merging cc calls and billings 

In [ ]:
# import pandas as pd
# import numpy as np
# import seaborn as sns
# import matplotlib.pyplot as plt
# import math
# import os
# from scipy.stats import pointbiserialr

# # ── YOUR EXISTING CODE ───────────────────────────────────────────────────────
# cc_calls    = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df = pd.read_csv('../../dataset/02_basic_clean/billings.csv')

# billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# cc_calls['Call_Date']                = pd.to_datetime(cc_calls['Call_Date'],                dayfirst=True)
# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(billings_df['Prospect_Renewal_Date'], dayfirst=True)

# merged_cc_calls = cc_calls.merge(billings_df, on='Co_Ref', how='inner')

# merged_cc_calls['Days_Before_Renewal'] = (
#     merged_cc_calls['Prospect_Renewal_Date'] - merged_cc_calls['Call_Date']
# ).dt.days

# merged_cc_calls = merged_cc_calls[
#     (merged_cc_calls['Days_Before_Renewal'] >= 14) &
#     (merged_cc_calls['Days_Before_Renewal'] <= 45)
# ]

# rows_to_drop = merged_cc_calls[merged_cc_calls['Prospect_Outcome'] == 'Open'].index
# merged_cc_calls.drop(rows_to_drop, inplace=True)

# merged_cc_calls = merged_cc_calls.sort_values(
#     by=['Co_Ref', 'Call_Date'], ascending=[True, True]
# )
# # ─────────────────────────────────────────────────────────────────────────────

# print('Shape after merge + filter:', merged_cc_calls.shape)
# print('Prospect_Outcome counts:')
# print(merged_cc_calls['Prospect_Outcome'].value_counts())

## Merging  bill + cc_calls (inner)

In [ ]:
# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# cc_calls = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df = pd.read_csv('../../dataset/02_basic_clean/billings.csv')

# # Keep only required billing columns (1 row per Co_Ref)
# billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# # Convert dates
# cc_calls['Call_Date'] = pd.to_datetime(cc_calls['Call_Date'], dayfirst=True, errors='coerce')
# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # ── STEP 1: MERGE (BILLING AS BASE) ─────────────────────────────────
# merged = billings_df.merge(cc_calls, on='Co_Ref', how='inner')

# print("Shape after merge:", merged.shape)

# # ── STEP 2: CALCULATE DAYS BEFORE RENEWAL ───────────────────────────
# merged['Days_Before_Renewal'] = (
#     merged['Prospect_Renewal_Date'] - merged['Call_Date']
# ).dt.days

# # ── STEP 3: APPLY 14–45 DAY FILTER ──────────────────────────────────
# filtered = merged[
#     (merged['Days_Before_Renewal'] >= 14) &
#     (merged['Days_Before_Renewal'] <= 45)
# ]

# print("Shape after 14–45 day filter:", filtered.shape)

# # ── STEP 4: SORT FOR CLEAR VIEW ─────────────────────────────────────
# filtered = filtered.sort_values(by=['Co_Ref', 'Call_Date'])

# # ── STEP 5: SHOW MULTIPLE CALLS PER CUSTOMER ────────────────────────
# print("\nExample (1 billing → multiple filtered calls):\n")

# for i, (coref, group) in enumerate(filtered.groupby('Co_Ref')):
#     print(f"Co_Ref: {coref}")
    
#     print("Billing Info:")
#     print(group[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']].iloc[0])
    
#     print("\nFiltered Calls (14–45 days before renewal):")
#     print(group[['Call_Date', 'Days_Before_Renewal']])
    
#     print("-" * 50)
    
#     if i == 2:
#         break

# # ── STEP 6: COUNT FILTERED CALLS ────────────────────────────────────
# call_counts = filtered.groupby('Co_Ref')['Call_Date'].count()

# print("\nTop customers by number of filtered calls:")
# print(call_counts.sort_values(ascending=False).head(10))

# # ── STEP 7: CUSTOMERS WITH NO CALLS IN WINDOW ───────────────────────
# customers_with_calls = filtered['Co_Ref'].nunique()
# total_customers = billings_df['Co_Ref'].nunique()

# print(f"\nCustomers with calls in 14–45 day window: {customers_with_calls}")
# print(f"Customers with NO calls in this window: {total_customers - customers_with_calls}")

# call_distribution = call_counts.value_counts().sort_index()

# print("\nDistribution of number of calls per customer:\n")

# for calls, num_customers in call_distribution.items():
#     print(f"{calls} calls → {num_customers} customers")

# print(filtered)


## Merging  bill + renew (inner)

In [10]:
import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# Convert dates
renewal_calls['Call_Date'] = pd.to_datetime(renewal_calls['Call_Date'], dayfirst=True, errors='coerce')
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Remove invalid calls
renewal_calls = renewal_calls.dropna(subset=['Call_Date'])

# ── MERGE ───────────────────────────────────────────────────────────
merged = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# Keep only calls BEFORE renewal
merged = merged[
    (merged['Call_Date'].isna()) |
    (merged['Call_Date'] <= merged['Prospect_Renewal_Date'])
]

# Days before renewal
merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

# Flag 14–45 calls
merged['renewal_14_45_flag'] = (
    (merged['Days_Before_Renewal'] >= 14) &
    (merged['Days_Before_Renewal'] <= 45)
).astype(int)

# ────────────────────────────────────────────────────────────────────
# ✅ AGGREGATION (COUNT PER CUSTOMER)
# ────────────────────────────────────────────────────────────────────
agg_calls = merged.groupby('Co_Ref').agg(
    renewal_call_count_14_45=('renewal_14_45_flag', 'sum')
).reset_index()

# ────────────────────────────────────────────────────────────────────
# ✅ MERGE BACK WITH FULL BILLING (ALL COLUMNS)
# ────────────────────────────────────────────────────────────────────
final_df = billings_df.merge(agg_calls, on='Co_Ref', how='left')

# Fill missing (customers with no calls)
final_df['renewal_call_count_14_45'] = final_df['renewal_call_count_14_45'].fillna(0)

# ── OUTPUT ──────────────────────────────────────────────────────────
print("\nFinal dataset shape:", final_df.shape)

print("\nSample data:")
print(final_df.head())

# Save
final_df.to_csv("../../dataset/05_merged/br_merged.csv", index=False)

C:\Users\NidharsanVelmurugan\AppData\Local\Temp\ipykernel_15232\3045362950.py:4: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Benefits_Mentioned, 20: Topic_Introduced_By, 21: Percentage_Price_Increase_Mentioned, 22: Monetary_Price_Increase_Mentioned, 23: Price_Range_Mentioned, 24: Customer_Asked_For_Justification, 25: Customer_Response, 26: Desire_To_Cancel, 27: Discount_Offered) have mixed types. Specify dtype option on im


Final dataset shape: (122082, 55)

Sample data:
   Co_Ref Renewal_Month Discount_Amount  Sustainability_Score  \
0  VT6174    01-11-2024               0                   8.0   
1  VD3828    01-08-2025               0                   8.0   
2  DV8120    01-03-2025               0                   8.0   
3  EZ9894    01-06-2025               0                   9.5   
4  FA8957    01-03-2025               0                   9.5   

   Total_Renewal_Score_New  Last_Years_Price  Auto_Renewal_Score  \
0                     42.5             799.0                   9   
1                     41.5             799.0                   9   
2                     33.0             799.0                   8   
3                     44.5             799.0                   9   
4                     42.5             799.0                   9   

   Status_Scores  Anchoring_Score  Tenure_Scores  ... Tenure_Group  \
0              9              7.5            9.0  ...            3   
1          

In [11]:
import pandas as pd

# ── SAMPLE DATA ─────────────────────────────────────────────────────
billings_df = pd.DataFrame({
    'Co_Ref': ['A', 'A'],
    'Prospect_Renewal_Date': ['12-01-2025', '12-01-2026']
})

renewal_calls = pd.DataFrame({
    'Co_Ref': ['A','A','A','A','A','A'],
    'Call_Date': [
        '19-12-2024',
        '20-12-2024',
        '21-12-2024',
        '19-12-2025',
        '19-12-2025',
        '19-12-2025'
    ]
})

# ── CONVERT DATES ───────────────────────────────────────────────────
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True
)

renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True
)

# ── MERGE ───────────────────────────────────────────────────────────
merged = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# Keep only calls BEFORE renewal
merged = merged[
    (merged['Call_Date'].isna()) |
    (merged['Call_Date'] <= merged['Prospect_Renewal_Date'])
]

# Days before renewal
merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

# Flag 14–45 calls
merged['renewal_14_45_flag'] = (
    (merged['Days_Before_Renewal'] >= 14) &
    (merged['Days_Before_Renewal'] <= 45)
).astype(int)

# ── AGGREGATION (ONLY BY Co_Ref - same as your code) ────────────────
agg_calls = merged.groupby('Co_Ref').agg(
    renewal_call_count_14_45=('renewal_14_45_flag', 'sum')
).reset_index()

# ── MERGE BACK ──────────────────────────────────────────────────────
final_df = billings_df.merge(agg_calls, on='Co_Ref', how='left')

# Fill missing
final_df['renewal_call_count_14_45'] = final_df['renewal_call_count_14_45'].fillna(0)

# ── OUTPUT ─────────────────────────────────────────────────────────
print(final_df)

  Co_Ref Prospect_Renewal_Date  renewal_call_count_14_45
0      A            2025-01-12                         6
1      A            2026-01-12                         6


# Merged  (billing + renew (inner ) + cc (left))


### inner cc
Final dataset shape: (27197, 73)

### left cc 
Final dataset shape: (27197, 73)

In [ ]:
import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# Keep required columns
billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# ── DATE CONVERSION ─────────────────────────────────────────────────
renewal_calls['Call_Date'] = pd.to_datetime(renewal_calls['Call_Date'], dayfirst=True, errors='coerce')
cc_calls['Call_Date']      = pd.to_datetime(cc_calls['Call_Date'], dayfirst=True, errors='coerce')
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Remove invalid dates
renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 1: BILLING + RENEWAL (INNER JOIN)
# ────────────────────────────────────────────────────────────────────
x = billings_df.merge(renewal_calls, on='Co_Ref', how='left')
# x = billings_df.merge(renewal_calls, on='Co_Ref', how='inner')

print("Shape after billing + renewal (INNER):", x.shape)

# Keep only calls BEFORE renewal
x = x[x['Call_Date'] <= x['Prospect_Renewal_Date']]

# Days before renewal
x['Days_Before_Renewal'] = (
    x['Prospect_Renewal_Date'] - x['Call_Date']
).dt.days

# Apply 14–45 filter (renewal calls)
x = x[
    (x['Days_Before_Renewal'] >= 14) &
    (x['Days_Before_Renewal'] <= 45)
]

print("Shape after renewal 14–45 filter:", x.shape)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 2: ADD CC CALLS (LEFT JOIN)
# ────────────────────────────────────────────────────────────────────
merged_cc = x.merge(cc_calls, on='Co_Ref', how='left', suffixes=('_renewal', '_cc'))

print("Shape after adding cc_calls (LEFT):", merged_cc.shape)

# ── FILTER CC CALLS ─────────────────────────────────────────────────

# Keep cc calls before renewal OR no call
merged_cc = merged_cc[
    (merged_cc['Call_Date_cc'].isna()) |
    (merged_cc['Call_Date_cc'] <= merged_cc['Prospect_Renewal_Date'])
]

# Calculate cc days before renewal
merged_cc['cc_Days_Before_Renewal'] = (
    merged_cc['Prospect_Renewal_Date'] - merged_cc['Call_Date_cc']
).dt.days

# Apply 14–45 filter to cc calls ALSO
merged_cc = merged_cc[
    (merged_cc['cc_Days_Before_Renewal'].isna()) |
    (
        (merged_cc['cc_Days_Before_Renewal'] >= 14) &
        (merged_cc['cc_Days_Before_Renewal'] <= 45)
    )
]

# Optional flag (still useful)
merged_cc['cc_14_45_flag'] = (
    (merged_cc['cc_Days_Before_Renewal'] >= 14) &
    (merged_cc['cc_Days_Before_Renewal'] <= 45)
).astype(int)

# ────────────────────────────────────────────────────────────────────
# 📊 FINAL OUTPUT
# ────────────────────────────────────────────────────────────────────
print("\nFinal dataset shape:", merged_cc.shape)

print("\nSample data after full merge:")
print(merged_cc.head())

# Optional: check distribution
print("\nUnique customers:", merged_cc['Co_Ref'].nunique())
print("\nOutcome distribution:")
print(merged_cc['Prospect_Outcome'].value_counts())









# //NEED TO DO AGG

merged_cc.to_csv("../../dataset/05_merged/brc_merged.csv")


C:\Users\NidharsanVelmurugan\AppData\Local\Temp\ipykernel_15232\1329530940.py:4: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Benefits_Mentioned, 20: Topic_Introduced_By, 21: Percentage_Price_Increase_Mentioned, 22: Monetary_Price_Increase_Mentioned, 23: Price_Range_Mentioned, 24: Customer_Asked_For_Justification, 25: Customer_Response, 26: Desire_To_Cancel, 27: Discount_Offered) have mixed types. Specify dtype option on im

Shape after billing + renewal (INNER): (441090, 38)
Shape after renewal 14–45 filter: (39837, 39)
Shape after adding cc_calls (LEFT): (57330, 71)

Final dataset shape: (27197, 73)

Sample data after full merge:
    Co_Ref Prospect_Renewal_Date Prospect_Outcome       Call_ID  \
0   VT6174            2024-11-05              Won  5.640000e+11   
1   VD3828            2025-08-09              Won  6.780000e+11   
5   EZ9894            2025-06-29              Won  6.510000e+11   
13  QS2598            2025-06-22              Won  6.510000e+11   
14  CJ9355            2025-03-17              Won  5.960000e+11   

   Call_Direction Call_Date_renewal Churn_Category Complaint_Category  \
0        Outbound        2024-10-09            NaN                NaN   
1       OUT_BOUND        2025-07-15            NaN                NaN   
5       OUT_BOUND        2025-06-02            NaN                NaN   
13      OUT_BOUND        2025-05-30            NaN                NaN   
14      OUT_BOUND    

# Merged  (billing + renew (inner ) + cc (left)) + email


In [6]:
import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
emails        = pd.read_csv('../../dataset/02_basic_clean/emails.csv')
billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# Keep required columns
billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# ── DATE CONVERSION ─────────────────────────────────────────────────
renewal_calls['Call_Date'] = pd.to_datetime(renewal_calls['Call_Date'], dayfirst=True, errors='coerce')
cc_calls['Call_Date']      = pd.to_datetime(cc_calls['Call_Date'], dayfirst=True, errors='coerce')
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Clean
renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 1: BILLING + RENEWAL (LEFT JOIN)
# ────────────────────────────────────────────────────────────────────
x = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# Keep only valid calls
x = x[x['Call_Date'] <= x['Prospect_Renewal_Date']]

# Days before renewal
x['Days_Before_Renewal'] = (
    x['Prospect_Renewal_Date'] - x['Call_Date']
).dt.days

# Apply 14–45 filter
x = x[
    (x['Days_Before_Renewal'] >= 14) &
    (x['Days_Before_Renewal'] <= 45)
]

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 2: ADD CC CALLS
# ────────────────────────────────────────────────────────────────────
merged_cc = x.merge(cc_calls, on='Co_Ref', how='left', suffixes=('_renewal', '_cc'))

# Keep valid cc calls
merged_cc = merged_cc[
    (merged_cc['Call_Date_cc'].isna()) |
    (merged_cc['Call_Date_cc'] <= merged_cc['Prospect_Renewal_Date'])
]

# cc timing
merged_cc['cc_Days_Before_Renewal'] = (
    merged_cc['Prospect_Renewal_Date'] - merged_cc['Call_Date_cc']
).dt.days

# 14–45 filter
merged_cc = merged_cc[
    (merged_cc['cc_Days_Before_Renewal'].isna()) |
    (
        (merged_cc['cc_Days_Before_Renewal'] >= 14) &
        (merged_cc['cc_Days_Before_Renewal'] <= 45)
    )
]

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 3: EMAIL PROCESSING
# ────────────────────────────────────────────────────────────────────

# ✔ Filter only 14 day emails
email_filter = emails[emails['Time_to_Renewal'] == '14_out']

# ✔ Aggregate emails per customer
email_agg = email_filter.groupby('Co_Ref').agg(
    email_count=('Co_Ref', 'count')
).reset_index()

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 4: MERGE EMAILS
# ────────────────────────────────────────────────────────────────────
final_df = merged_cc.merge(email_agg, on='Co_Ref', how='left')

# Fill missing emails
final_df['email_count'] = final_df['email_count'].fillna(0)

# ────────────────────────────────────────────────────────────────────
# 📊 OUTPUT
# ────────────────────────────────────────────────────────────────────
print("\nFinal dataset shape:", final_df.shape)

print("\nSample data:")
print(final_df.head())

print("\nUnique customers:", final_df['Co_Ref'].nunique())

print("\nOutcome distribution:")
print(final_df['Prospect_Outcome'].value_counts())




# //NEED TO AGGREGATE


final_df.to_csv("../../dataset/05_merged/brce_merged.csv")


C:\Users\NidharsanVelmurugan\AppData\Local\Temp\ipykernel_15232\1969126664.py:4: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Benefits_Mentioned, 20: Topic_Introduced_By, 21: Percentage_Price_Increase_Mentioned, 22: Monetary_Price_Increase_Mentioned, 23: Price_Range_Mentioned, 24: Customer_Asked_For_Justification, 25: Customer_Response, 26: Desire_To_Cancel, 27: Discount_Offered) have mixed types. Specify dtype option on im


Final dataset shape: (27197, 73)

Sample data:
   Co_Ref Prospect_Renewal_Date Prospect_Outcome       Call_ID Call_Direction  \
0  VT6174            2024-11-05              Won  5.640000e+11       Outbound   
1  VD3828            2025-08-09              Won  6.780000e+11      OUT_BOUND   
2  EZ9894            2025-06-29              Won  6.510000e+11      OUT_BOUND   
3  QS2598            2025-06-22              Won  6.510000e+11      OUT_BOUND   
4  CJ9355            2025-03-17              Won  5.960000e+11      OUT_BOUND   

  Call_Date_renewal Churn_Category Complaint_Category  \
0        2024-10-09            NaN                NaN   
1        2025-07-15            NaN                NaN   
2        2025-06-02            NaN                NaN   
3        2025-05-30            NaN                NaN   
4        2025-02-18            NaN                NaN   

  Customer_Reaction_Category Agent_Renewal_Pitch_Category  ...  \
0                        NaN             Expiration / Du